## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "downstr.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR = Path(DATA_ROOT, "notebooks/3UTR/Wildtype_mice/2026")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
import os
from collections import defaultdict

from sklearn.metrics import silhouette_score
import numpy as np
import pandas as pd
import scanpy as sc

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.sparse import issparse

# Load data


In [ ]:
BASEPATH = f"{DATA_ROOT}/3utr/wildtypes_analysis/2026_analyses/utrcalling"

In [ ]:
adata_counts = sc.read_h5ad(
    os.path.join(BASEPATH, "UTR_count_table_genelevel_bw12_above10cts_apa_named.h5ad")
)

adata_counts

In [ ]:
adata_counts.obs["batch_number"] = pd.Categorical(
    adata_counts.obs["batch_number"].astype(int),
    categories=sorted(adata_counts.obs["batch_number"].astype(int).unique()),
    ordered=True,
)

In [ ]:
lineage_celltypes = [
    "Astrocytes",
    "qNSC1",
    "qNSC2",
    "ROB_qNSC2",
    "aNSC1",
    "aNSC2",
    "TAP",
    "NB",
    "Neurons",
]

# Convert raw counts to within-gene proportions

For each gene (grouped by `adata.var["name"]`), divide each UTR site's raw count by the
gene's total count per cell. This yields values in [0, 1] that sum to 1 within each gene
per cell. Cells where a gene has zero total counts receive 0 for all its sites.
Single-site genes are excluded — their proportion is always 1 and carries no
isoform-switching information.


In [ ]:
# Build gene -> site index mapping; keep only multi-site genes
gene_to_site_idx = defaultdict(list)
for i, gene_id in enumerate(adata_counts.var["name"]):
    gene_to_site_idx[gene_id].append(i)

multi_site_genes = {g: s for g, s in gene_to_site_idx.items() if len(s) >= 2}
multi_site_global_idx = sorted(
    idx for sites in multi_site_genes.values() for idx in sites
)

print(f"Total genes:        {len(gene_to_site_idx)}")
print(f"Multi-site genes:   {len(multi_site_genes)}")
print(f"Single-site genes:  {len(gene_to_site_idx) - len(multi_site_genes)}")
print(f"UTR sites retained: {len(multi_site_global_idx)}")

In [ ]:
# Compute within-gene proportions per cell
X = adata_counts.X
X_dense = (
    X.toarray().astype(np.float64) if issparse(X) else np.asarray(X, dtype=np.float64)
)

# Subset to multi-site features
X_multi = X_dense[:, multi_site_global_idx]
multi_site_var = adata_counts.var.iloc[multi_site_global_idx].copy()

# Build local gene -> column mapping
gene_to_local = defaultdict(list)
for local_i, global_i in enumerate(multi_site_global_idx):
    gene_id = adata_counts.var.iloc[global_i]["name"]
    gene_to_local[gene_id].append(local_i)

# Compute proportions
X_prop = np.zeros_like(X_multi, dtype=np.float32)
for gene_id, local_cols in gene_to_local.items():
    gene_counts = X_multi[:, local_cols]
    gene_total = gene_counts.sum(axis=1, keepdims=True)
    with np.errstate(invalid="ignore", divide="ignore"):
        props = np.where(gene_total > 0, gene_counts / gene_total, 0.0)
    X_prop[:, local_cols] = props

print(f"Proportions matrix shape: {X_prop.shape}")
print(f"Value range: [{X_prop.min():.3f}, {X_prop.max():.3f}]")

In [ ]:
# Validate: per-gene proportions should sum to 0 (unexpressed) or 1 per cell
errors = []
for gene_id, local_cols in gene_to_local.items():
    row_sums = X_prop[:, local_cols].sum(axis=1)
    invalid = ~(
        np.isclose(row_sums, 0.0, atol=1e-3) | np.isclose(row_sums, 1.0, atol=1e-3)
    )
    if invalid.any():
        errors.append((gene_id, int(invalid.sum()), row_sums[invalid][:3]))

if errors:
    print(f"\u274c {len(errors)} genes with invalid sums:")
    for g, n, ex in errors[:5]:
        print(f"  {g}: {n} cells, examples: {ex}")
else:
    print("\u2705 All per-gene proportions sum to 0 or 1.")

# Summary
all_sums = np.concatenate(
    [X_prop[:, cols].sum(axis=1) for cols in gene_to_local.values()]
)
zero_frac = np.isclose(all_sums, 0.0, atol=1e-3).mean()
one_frac = np.isclose(all_sums, 1.0, atol=1e-3).mean()
print(f"\nSummary over all (gene, cell) pairs:")
print(f"  Total pairs:          {len(all_sums):,}")
print(f"  Sum == 0 (no counts): {zero_frac:.2%}")
print(f"  Sum == 1 (valid):     {one_frac:.2%}")

In [ ]:
# Build AnnData with proportions
adata_prop = sc.AnnData(
    X=X_prop,
    obs=adata_counts.obs.copy(),
    var=multi_site_var,
)

print(adata_prop)

In [ ]:
adata_prop2 = sc.read_h5ad(
    os.path.join(
        BASEPATH, "UTR_count_table_genelevel_bw12_above10cts_apa_named_proportions.h5ad"
    )
)

adata_prop2

In [ ]:
# ── Compare adata_prop vs adata_prop2 ────────────────────────────────────────
MAX_SHOW = 10  # max differing rows/cols to display

# ── .obs ─────────────────────────────────────────────────────────────────────
obs_cols_same = list(adata_prop.obs.columns) == list(adata_prop2.obs.columns)
if not obs_cols_same:
    print("obs columns differ!")
    print("  adata_prop  :", list(adata_prop.obs.columns))
    print("  adata_prop2 :", list(adata_prop2.obs.columns))
else:
    obs_diff = ~(adata_prop.obs == adata_prop2.obs)
    n_obs_diff = obs_diff.values.sum()
    if n_obs_diff == 0:
        print("✅ .obs  — identical")
    else:
        print(f"❌ .obs  — {n_obs_diff} differing cell(s)")
        diff_mask = obs_diff.any(axis=1)
        display(adata_prop.obs[diff_mask].head(MAX_SHOW))
        print("--- adata_prop2 ---")
        display(adata_prop2.obs[diff_mask].head(MAX_SHOW))

# ── .var ─────────────────────────────────────────────────────────────────────
var_cols_same = list(adata_prop.var.columns) == list(adata_prop2.var.columns)
if not var_cols_same:
    print("\nvar columns differ!")
    print("  adata_prop  :", list(adata_prop.var.columns))
    print("  adata_prop2 :", list(adata_prop2.var.columns))
else:
    var_diff = ~(adata_prop.var == adata_prop2.var)
    n_var_diff = var_diff.values.sum()
    if n_var_diff == 0:
        print("✅ .var  — identical")
    else:
        print(f"\n❌ .var  — {n_var_diff} differing feature(s)")
        diff_mask_v = var_diff.any(axis=1)
        display(adata_prop.var[diff_mask_v].head(MAX_SHOW))
        print("--- adata_prop2 ---")
        display(adata_prop2.var[diff_mask_v].head(MAX_SHOW))

# ── .X ───────────────────────────────────────────────────────────────────────
X1 = adata_prop.X.toarray() if issparse(adata_prop.X) else np.asarray(adata_prop.X)
X2 = adata_prop2.X.toarray() if issparse(adata_prop2.X) else np.asarray(adata_prop2.X)

if X1.shape != X2.shape:
    print(f"\n❌ .X   — shape mismatch: {X1.shape} vs {X2.shape}")
else:
    x_diff_mask = ~np.isclose(X1, X2, atol=1e-6, equal_nan=True)
    n_x_diff = x_diff_mask.sum()
    if n_x_diff == 0:
        print("✅ .X   — identical (within atol=1e-6)")
    else:
        rows, cols = np.where(x_diff_mask)
        print(
            f"\n❌ .X   — {n_x_diff} differing element(s)  "
            f"(max abs diff: {np.abs(X1 - X2)[x_diff_mask].max():.6g})"
        )
        show = min(MAX_SHOW, len(rows))
        df_x = pd.DataFrame(
            {
                "cell": adata_prop.obs_names[rows[:show]],
                "feature": adata_prop.var_names[cols[:show]],
                "adata_prop": X1[rows[:show], cols[:show]],
                "adata_prop2": X2[rows[:show], cols[:show]],
                "diff": (X1 - X2)[rows[:show], cols[:show]],
            }
        )
        print(df_x.to_string(index=False))

# Preprocess and integrate

Preprocessing for proportions differs from the standard counts pipeline:

- **Skip `normalize_total`** — proportions aren't counts; library-size normalization doesn't apply
- **Skip `log1p`** — proportions are bounded [0, 1]; log transform is designed for count data
- **Apply `sc.pp.scale()`** — z-score standardization (center + unit variance) for PCA


In [ ]:
# Shared preprocessing: scale -> PCA (no normalize_total or log1p)
adata_pp = adata_prop.copy()
adata_pp.raw = adata_pp.copy()  # store raw proportions before scaling

sc.pp.scale(adata_pp, max_value=10)

sc.tl.pca(adata_pp, svd_solver="arpack", n_comps=50)
sc.pl.pca_variance_ratio(adata_pp, log=True, n_pcs=50)

print(adata_pp)

## 1. Harmony

**Harmony** iteratively adjusts the PCA embedding to remove batch effects. Fast, widely
used, and integrates tightly with Scanpy. Produces `X_pca_harmony`.


In [ ]:
import harmonypy as hm

adata_harmony = adata_pp.copy()
X_pca = adata_harmony.obsm["X_pca"].astype(np.float64)

ho = hm.run_harmony(X_pca, adata_harmony.obs, "batch_number")

Z = ho.Z_corr
# Ensure shape is (n_obs, n_components) regardless of harmonypy version
if Z.shape[0] != adata_harmony.n_obs:
    Z = Z.T
adata_harmony.obsm["X_pca_harmony"] = Z

sc.pp.neighbors(adata_harmony, use_rep="X_pca_harmony", n_neighbors=10, n_pcs=40)
sc.tl.umap(adata_harmony)
sc.tl.leiden(adata_harmony)

adata_harmony

## 2. Scanorama

Uses **mutual nearest neighbours across batches** in PCA space to produce a joint
low-dimensional corrected embedding (`X_scanorama`). Handles situations where not all
cell types are present in every batch.


In [ ]:
# Scanorama requires batches to be contiguous in the AnnData object.
adata_scanorama = adata_pp[adata_pp.obs.sort_values("batch_number").index].copy()
sc.external.pp.scanorama_integrate(
    adata_scanorama,
    key="batch_number",
    basis="X_pca",
    adjusted_basis="X_scanorama",
)
sc.pp.neighbors(adata_scanorama, use_rep="X_scanorama", n_neighbors=10)
sc.tl.umap(adata_scanorama)
sc.tl.leiden(adata_scanorama)

adata_scanorama

## 3. BBKNN

**Batch-Balanced KNN** replaces the `sc.pp.neighbors` step with a batch-aware graph.
Fast and often effective. Does not produce a corrected embedding — only the graph (and
therefore UMAP) is batch-corrected.


In [ ]:
adata_bbknn = adata_pp.copy()
sc.external.pp.bbknn(adata_bbknn, batch_key="batch_number", n_pcs=40)
sc.tl.umap(adata_bbknn)
sc.tl.leiden(adata_bbknn)

adata_bbknn

## 4. ComBat

Corrects batch effects at the **expression level** (before scaling/PCA). Applied to the
raw proportions before z-scoring — more aggressive than embedding-level methods.


In [ ]:
# ComBat on raw proportions -> then scale -> PCA
adata_combat = adata_prop.copy()
adata_combat.raw = adata_combat

sc.pp.combat(adata_combat, key="batch_number")

sc.pp.scale(adata_combat, max_value=10)
sc.tl.pca(adata_combat, svd_solver="arpack", n_comps=50)
sc.pp.neighbors(adata_combat, n_neighbors=10, n_pcs=40)
sc.tl.umap(adata_combat)
sc.tl.leiden(adata_combat)

adata_combat

# Dimensionality reduction


## Baseline (no integration)

PCA and UMAP on the preprocessed proportions **without** any batch correction. Use this
as a visual reference against which each integration result can be judged.


In [ ]:
adata_base = adata_pp.copy()
sc.pp.neighbors(adata_base, n_neighbors=10, n_pcs=40)
sc.tl.umap(adata_base)
sc.tl.leiden(adata_base)

adata_base

In [ ]:
sc.pl.pca(
    adata_base,
    color=[
        "batch_number",
        "celltype",
        "leiden",
        "dpt_pseudotime",
        "phase",
        "n_genes_by_counts",
    ],
    wspace=0.4,
    ncols=2,
)

In [ ]:
sc.pl.umap(
    adata_base,
    color=[
        "batch_number",
        "celltype",
        "leiden",
        "dpt_pseudotime",
        "phase",
        "n_genes_by_counts",
    ],
    wspace=0.4,
    ncols=3,
    title=[
        "Baseline \u2014 batch",
        "Baseline \u2014 celltype",
        "Baseline \u2014 leiden",
        "Baseline \u2014 pseudotime",
        "Baseline \u2014 phase",
        "Baseline \u2014 n_genes",
    ],
)

## 1. Harmony


In [ ]:
# Lineage subset: exclude neurons that lack dpt_pseudotime
lineage_mask = adata_harmony.obs["celltype"].isin(lineage_celltypes)
neuron_no_pt = (adata_harmony.obs["celltype"] == "Neurons") & adata_harmony.obs[
    "dpt_pseudotime"
].isna()
lineage_mask = lineage_mask & ~neuron_no_pt

adata_harmony_lineage = adata_harmony[lineage_mask].copy()
print(f"Lineage cells: {adata_harmony_lineage.n_obs} / {adata_harmony.n_obs}")

In [ ]:
sc.pl.embedding(
    adata_harmony,
    basis="pca_harmony",
    color=[
        "leiden",
        "phase",
        "batch_number",
        "dpt_pseudotime",
        "n_genes_by_counts",
        "celltype",
    ],
    wspace=0.4,
    ncols=2,
)

In [ ]:
sc.pl.umap(
    adata_harmony,
    color=[
        "leiden",
        "phase",
        "batch_number",
        "dpt_pseudotime",
        "n_genes_by_counts",
        "celltype",
    ],
    wspace=0.4,
    ncols=2,
)

In [ ]:
sc.pl.umap(
    adata_harmony_lineage,
    color=[
        "leiden",
        "phase",
        "batch_number",
        "dpt_pseudotime",
        "n_genes_by_counts",
        "celltype",
    ],
    wspace=0.4,
    ncols=2,
)

In [ ]:
# Integration quality - Harmony
np.random.seed(42)
n_sample = min(5000, adata_harmony.shape[0])
idx_h = np.random.choice(adata_harmony.shape[0], n_sample, replace=False)

emb_h = adata_harmony.obsm["X_pca_harmony"][idx_h]
batch_h = adata_harmony.obs["batch_number"].astype(str).values[idx_h]
ct_h = adata_harmony.obs["celltype"].values[idx_h]

asw_batch_harmony = silhouette_score(emb_h, batch_h)
asw_ct_harmony = silhouette_score(emb_h, ct_h)

print(
    f"Harmony \u2014 ASW batch    (\u2193 = better batch mixing):       {asw_batch_harmony:.4f}"
)
print(
    f"Harmony \u2014 ASW celltype (\u2191 = better biology preserved):  {asw_ct_harmony:.4f}"
)

## 2. Scanorama


In [ ]:
lineage_mask_s = adata_scanorama.obs["celltype"].isin(lineage_celltypes)
neuron_no_pt_s = (adata_scanorama.obs["celltype"] == "Neurons") & adata_scanorama.obs[
    "dpt_pseudotime"
].isna()
lineage_mask_s = lineage_mask_s & ~neuron_no_pt_s

adata_scanorama_lineage = adata_scanorama[lineage_mask_s].copy()

In [ ]:
sc.pl.embedding(
    adata_scanorama,
    basis="scanorama",
    color=[
        "leiden",
        "phase",
        "batch_number",
        "dpt_pseudotime",
        "n_genes_by_counts",
        "celltype",
    ],
    wspace=0.4,
    ncols=2,
)

In [ ]:
sc.pl.umap(
    adata_scanorama,
    color=[
        "leiden",
        "phase",
        "batch_number",
        "dpt_pseudotime",
        "n_genes_by_counts",
        "celltype",
    ],
    wspace=0.4,
    ncols=2,
)

In [ ]:
sc.pl.umap(
    adata_scanorama_lineage,
    color=[
        "leiden",
        "phase",
        "batch_number",
        "dpt_pseudotime",
        "n_genes_by_counts",
        "celltype",
    ],
    wspace=0.4,
    ncols=2,
)

In [ ]:
idx_s = np.random.choice(adata_scanorama.shape[0], n_sample, replace=False)

emb_s = adata_scanorama.obsm["X_scanorama"][idx_s]
batch_s = adata_scanorama.obs["batch_number"].astype(str).values[idx_s]
ct_s = adata_scanorama.obs["celltype"].values[idx_s]

asw_batch_scanorama = silhouette_score(emb_s, batch_s)
asw_ct_scanorama = silhouette_score(emb_s, ct_s)

print(
    f"Scanorama \u2014 ASW batch    (\u2193 = better batch mixing):       {asw_batch_scanorama:.4f}"
)
print(
    f"Scanorama \u2014 ASW celltype (\u2191 = better biology preserved):  {asw_ct_scanorama:.4f}"
)

## 3. BBKNN


In [ ]:
lineage_mask_b = adata_bbknn.obs["celltype"].isin(lineage_celltypes)
neuron_no_pt_b = (adata_bbknn.obs["celltype"] == "Neurons") & adata_bbknn.obs[
    "dpt_pseudotime"
].isna()
lineage_mask_b = lineage_mask_b & ~neuron_no_pt_b

adata_bbknn_lineage = adata_bbknn[lineage_mask_b].copy()

In [ ]:
print("BBKNN: no corrected embedding available \u2014 see UMAP below.")

In [ ]:
sc.pl.umap(
    adata_bbknn,
    color=[
        "leiden",
        "phase",
        "batch_number",
        "dpt_pseudotime",
        "n_genes_by_counts",
        "celltype",
    ],
    wspace=0.4,
    ncols=2,
)

In [ ]:
sc.pl.umap(
    adata_bbknn_lineage,
    color=[
        "leiden",
        "phase",
        "batch_number",
        "dpt_pseudotime",
        "n_genes_by_counts",
        "celltype",
    ],
    wspace=0.4,
    ncols=2,
)

In [ ]:
# For BBKNN, correction is in the graph; we use the UMAP embedding as proxy for ASW.
idx_b = np.random.choice(adata_bbknn.shape[0], n_sample, replace=False)

emb_b = adata_bbknn.obsm["X_umap"][idx_b]
batch_b = adata_bbknn.obs["batch_number"].astype(str).values[idx_b]
ct_b = adata_bbknn.obs["celltype"].values[idx_b]

asw_batch_bbknn = silhouette_score(emb_b, batch_b)
asw_ct_bbknn = silhouette_score(emb_b, ct_b)

print(
    f"BBKNN \u2014 ASW batch    (\u2193 = better batch mixing):       {asw_batch_bbknn:.4f}"
)
print(
    f"BBKNN \u2014 ASW celltype (\u2191 = better biology preserved):  {asw_ct_bbknn:.4f}"
)

## 4. ComBat


In [ ]:
lineage_mask_c = adata_combat.obs["celltype"].isin(lineage_celltypes)
neuron_no_pt_c = (adata_combat.obs["celltype"] == "Neurons") & adata_combat.obs[
    "dpt_pseudotime"
].isna()
lineage_mask_c = lineage_mask_c & ~neuron_no_pt_c

adata_combat_lineage = adata_combat[lineage_mask_c].copy()

In [ ]:
sc.pl.pca(
    adata_combat,
    color=[
        "leiden",
        "phase",
        "batch_number",
        "dpt_pseudotime",
        "n_genes_by_counts",
        "celltype",
    ],
    wspace=0.4,
    ncols=2,
)

In [ ]:
sc.pl.umap(
    adata_combat,
    color=[
        "leiden",
        "phase",
        "batch_number",
        "dpt_pseudotime",
        "n_genes_by_counts",
        "celltype",
    ],
    wspace=0.4,
    ncols=2,
)

In [ ]:
sc.pl.umap(
    adata_combat_lineage,
    color=[
        "leiden",
        "phase",
        "batch_number",
        "dpt_pseudotime",
        "n_genes_by_counts",
        "celltype",
    ],
    wspace=0.4,
    ncols=2,
)

In [ ]:
idx_c = np.random.choice(adata_combat.shape[0], n_sample, replace=False)

emb_c = adata_combat.obsm["X_pca"][idx_c]
batch_c = adata_combat.obs["batch_number"].astype(str).values[idx_c]
ct_c = adata_combat.obs["celltype"].values[idx_c]

asw_batch_combat = silhouette_score(emb_c, batch_c)
asw_ct_combat = silhouette_score(emb_c, ct_c)

print(
    f"ComBat \u2014 ASW batch    (\u2193 = better batch mixing):       {asw_batch_combat:.4f}"
)
print(
    f"ComBat \u2014 ASW celltype (\u2191 = better biology preserved):  {asw_ct_combat:.4f}"
)

## Integration quality comparison

Side-by-side summary of the ASW metrics across all methods.

- **ASW batch** (x-axis left): lower / more negative = batches are better mixed (desired)
- **ASW celltype** (x-axis right): higher = biological structure is better preserved (desired)

Note: Harmony/Scanorama ASW is on their respective corrected embeddings; BBKNN on X_umap;
ComBat on the corrected PCA. Methods are not strictly comparable in absolute terms, but the
relative values within each metric are informative.


In [ ]:
results = pd.DataFrame(
    {
        "Method": ["Harmony", "Scanorama", "BBKNN*", "ComBat"],
        "ASW batch": [
            asw_batch_harmony,
            asw_batch_scanorama,
            asw_batch_bbknn,
            asw_batch_combat,
        ],
        "ASW celltype": [
            asw_ct_harmony,
            asw_ct_scanorama,
            asw_ct_bbknn,
            asw_ct_combat,
        ],
    }
).set_index("Method")

print(results.round(4))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

colors_bar = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

axes[0].barh(results.index, results["ASW batch"], color=colors_bar)
axes[0].axvline(0, color="k", linewidth=0.8, linestyle="--")
axes[0].set_xlabel("ASW batch  (\u2193 = better mixing)")
axes[0].set_title("Batch mixing")

axes[1].barh(results.index, results["ASW celltype"], color=colors_bar)
axes[1].axvline(0, color="k", linewidth=0.8, linestyle="--")
axes[1].set_xlabel("ASW celltype  (\u2191 = better biology)")
axes[1].set_title("Biology preservation")

fig.suptitle("Proportions integration quality (ASW scores)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# * BBKNN ASW is computed on X_umap (no corrected low-dim embedding is produced)

# Store data


In [ ]:
# Save Harmony-integrated proportions (all cells + lineage subset)
adata_harmony.write_h5ad(
    os.path.join(
        BASEPATH,
        "UTR_count_table_genelevel_bw12_above10cts_apa_named_adata_harmony_proportions.h5ad",
    )
)
adata_harmony_lineage.write_h5ad(
    os.path.join(
        BASEPATH,
        "UTR_count_table_genelevel_bw12_above10cts_apa_named_adata_harmony_proportions_lineage.h5ad",
    )
)

print(
    f"Saved all-cells:  ...adata_harmony_proportions.h5ad  ({adata_harmony.n_obs} cells)"
)
print(
    f"Saved lineage:    ...adata_harmony_proportions_lineage.h5ad  ({adata_harmony_lineage.n_obs} cells)"
)

In [ ]:
adata_harmony.raw.X